**ФИО студента:** [Иванов Иван Иванович]

**Задание:** `outpainting`

**Группа:** [номер группы]

**Дата сдачи:** [дата]


## Блок настройки окружения

Код для импорта библиотек, для формирования глобальных констант, других конфигураций (если требуется). Постарайтесь здесь зафиксировать все seed, которые сможете

In [ ]:
import random

import hashlib
from pathlib import Path
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Image, display
from torch.utils.data import DataLoader
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda" and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16

MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
MLFLOW_EXPERIMENT_NAME = "outpainting"
MLFLOW_MODEL_NAME = "outpainting_unet"

try:
    import mlflow
    from mlflow.exceptions import MlflowException
    from mlflow.tracking import MlflowClient

    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
    MLFLOW_AVAILABLE = True
except Exception:
    MLFLOW_AVAILABLE = False

print(f"Трекинг MLflow: {'включен' if MLFLOW_AVAILABLE else 'отключен'}")



## Блок "Данные"

Входной файл проверяется по SHA-256, форме и заполнению области пропусков нулями. Куб предзагружен в оперативную память.

Пять соседних срезов дают 2.5D-контекст. Патч 368×256 помещается между граничными полосами. Маска воспроизводит скрытые полосы шириной 24 сверху и снизу.

Для train динамически генерируются положение патча, отражения и масштаб амплитуд. Gaussian noise заранее записан в банк из 64 вариантов; пример использует вариант по своему индексу, поэтому генерация шума не выполняется в обучающем цикле. Validation и test фиксированы и не аугментируются.

Нормализация рассчитывается только по train-срезам. Между train, validation и test оставлены защитные интервалы, поэтому пятисрезовые контексты не пересекаются.


In [ ]:
from seismic_dataset import (
    AMP_SCALE_RANGE,
    NOISE_SIGMA,
    NOISE_BANK_SIZE,
    OUT_PAINT_MASK_BAND as MASK_W,
    OUT_PAINT_PATCH_H as PATCH_H,
    OUT_PAINT_PATCH_W as PATCH_W
)

DATA_PATH = Path("input_seismic_data/outpainting_cube_missed_data.npy")
EXPECTED_SHAPE = (460, 481, 800)
EXPECTED_SHA256 = "b33681e31f22c7cfa21de6b96f17e640a981f3ead16579b0656f063a4b9231fa"

sha256 = hashlib.sha256()
with DATA_PATH.open("rb") as file:
    for chunk in iter(lambda: file.read(8 * 1024 * 1024), b""):
        sha256.update(chunk)

actual_sha256 = sha256.hexdigest()
assert actual_sha256 == EXPECTED_SHA256, (
    f"SHA-256 не совпадает: ожидался {EXPECTED_SHA256}, получен {actual_sha256}"
)

cube = np.load(DATA_PATH)
assert cube.shape == EXPECTED_SHAPE
assert cube.dtype == np.float32
print(cube.shape)

real_mask = np.zeros(cube.shape, dtype=bool)
real_mask[:, :24, :] = True
real_mask[:, 457:481, :] = True

for slice_index in range(cube.shape[0]):
    assert np.all(cube[slice_index][real_mask[slice_index]] == 0), (
        f"Область пропусков в срезе {slice_index} содержит ненулевые значения"
    )

missing_count = int(real_mask.sum())
print(
    f"Проверка нулей пройдена: {missing_count:,} пропущенных значений, "
    f"{100 * missing_count / cube.size:.2f}% куба"
)

assert 457 - 24 >= PATCH_H
assert cube.shape[2] >= PATCH_W
assert PATCH_H % 16 == 0 and PATCH_W % 16 == 0
assert 2 * MASK_W < PATCH_H


In [ ]:
from seismic_dataset import OutpaintingDataset

train_centers = np.arange(2, 322)
val_centers = np.arange(326, 396)
test_centers = np.arange(400, 458)
TRAIN_SAMPLES_PER_CENTER = 4


In [ ]:
train_sample_centers = train_centers[::4]
sample = cube[train_sample_centers, ::4, ::4]
sample_mask = real_mask[train_sample_centers, ::4, ::4]
sample_known = ~sample_mask
mean = float(sample[sample_known].mean())
std = max(float(sample[sample_known].std()), 1e-6)

train_dataset = OutpaintingDataset(
    cube, train_centers, TRAIN_SAMPLES_PER_CENTER, mean, std, True, 42, True
)
val_dataset = OutpaintingDataset(cube, val_centers, 4, mean, std, False, 43, False)
test_dataset = OutpaintingDataset(cube, test_centers, 4, mean, std, False, 44, False)


In [ ]:
BATCH_SIZE = 16
NUM_WORKERS = 0

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0
)


## Блок "Модель"

Используется 2.5D U-Net с `base_channels = 24`: пять соседних срезов и маска подаются как шесть каналов. Bottleneck содержит свёртки с dilation 2 и 4. Размер батча равен 16.


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(8, out_channels),
            nn.GELU(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(8, out_channels),
            nn.GELU()
        )

    def forward(self, x):
        return self.block(x)


class DilatedBottleneck(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(8, out_channels),
            nn.GELU(),
            nn.Conv2d(
                out_channels,
                out_channels,
                3,
                padding=2,
                dilation=2,
                bias=False
            ),
            nn.GroupNorm(8, out_channels),
            nn.GELU(),
            nn.Conv2d(
                out_channels,
                out_channels,
                3,
                padding=4,
                dilation=4,
                bias=False
            ),
            nn.GroupNorm(8, out_channels),
            nn.GELU()
        )

    def forward(self, x):
        return self.block(x)


In [ ]:
class UNet25D(nn.Module):
    def __init__(self, in_channels=6, out_channels=1, base_channels=24):
        super().__init__()
        self.pool = nn.AvgPool2d(2)
        self.encoders = nn.ModuleList([
            ConvBlock(in_channels, base_channels),
            ConvBlock(base_channels, base_channels * 2),
            ConvBlock(base_channels * 2, base_channels * 4),
            ConvBlock(base_channels * 4, base_channels * 8)
        ])
        self.bottleneck = DilatedBottleneck(base_channels * 8, base_channels * 16)
        self.upconvs = nn.ModuleList([
            nn.Conv2d(base_channels * 16, base_channels * 8, 3, padding=1),
            nn.Conv2d(base_channels * 8, base_channels * 4, 3, padding=1),
            nn.Conv2d(base_channels * 4, base_channels * 2, 3, padding=1),
            nn.Conv2d(base_channels * 2, base_channels, 3, padding=1)
        ])
        self.decoders = nn.ModuleList([
            ConvBlock(base_channels * 16, base_channels * 8),
            ConvBlock(base_channels * 8, base_channels * 4),
            ConvBlock(base_channels * 4, base_channels * 2),
            ConvBlock(base_channels * 2, base_channels)
        ])
        self.head = nn.Conv2d(base_channels, out_channels, 1)

    def forward(self, x):
        height, width = x.shape[-2:]
        pad_height = (-height) % 16
        pad_width = (-width) % 16
        x = F.pad(x, (0, pad_width, 0, pad_height))

        skips = []
        for encoder in self.encoders:
            x = encoder(x)
            skips.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)

        for upconv, decoder, skip in zip(
            self.upconvs,
            self.decoders,
            reversed(skips)
        ):
            x = F.interpolate(x, scale_factor=2, mode="nearest")
            x = upconv(x)
            x = torch.cat((x, skip), dim=1)
            x = decoder(x)

        x = self.head(x)
        return x[..., :height, :width]


model = UNet25D()
print(f"Параметров модели: {sum(p.numel() for p in model.parameters()):,}")


## Блок "Обучение модели"

Потеря объединяет маскированные L1, MSE, gradient L1 и `0.5 · (1 − R)`. High-frequency и Fourier/FFL-слагаемые удалены. Норма градиентов не вычисляется и не логируется. Лучшая модель выбирается по глобальному validation R.

Используются `BATCH_SIZE = 16`, `EPOCHS = 75`, `GRAD_LOSS_WEIGHT = 0.3`, `R_LOSS_WEIGHT = 0.5`, AdamW с LR `2e-3` и `ReduceLROnPlateau`. Восстановление чекпоинта включает early stopping и состояния генераторов случайных чисел.


In [ ]:
def masked_mse(prediction, target, mask):
    error = (prediction - target) ** 2
    return (error * mask).sum() / mask.sum().clamp_min(1)


def masked_l1(prediction, target, mask):
    error = torch.abs(prediction - target)
    return (error * mask).sum() / mask.sum().clamp_min(1)


def masked_gradient_l1(prediction, target, mask):
    mask_x = mask[..., 1:] * mask[..., :-1]
    mask_y = mask[..., 1:, :] * mask[..., :-1, :]
    error_x = (prediction[..., 1:] - prediction[..., :-1] - target[..., 1:] + target[..., :-1]).abs()
    error_y = (
        prediction[..., 1:, :] - prediction[..., :-1, :]
        - target[..., 1:, :] + target[..., :-1, :]
    ).abs()
    return (
        (error_x * mask_x).sum() + (error_y * mask_y).sum()
    ) / (mask_x.sum() + mask_y.sum()).clamp_min(1)


def correlation_r(prediction, target):
    prediction = prediction - prediction.mean()
    target = target - target.mean()
    denominator = prediction.norm() * target.norm()
    return (prediction * target).sum() / denominator.clamp_min(1e-8)


def masked_r(prediction, target, mask):
    return correlation_r(
        prediction[mask.bool()].float(),
        target[mask.bool()].float(),
    )


def reconstruction_loss(prediction, target, mask, return_components=False):
    l1_loss = masked_l1(prediction, target, mask)
    mse_loss = masked_mse(prediction, target, mask)
    r_loss = 1 - masked_r(prediction, target, mask)
    components = {
        "l1_loss": L1_WEIGHT * l1_loss,
        "mse_loss": mse_loss,
        "r_loss": R_LOSS_WEIGHT * r_loss,
        "gradient_loss": prediction.new_zeros(()),
    }
    loss = components["l1_loss"] + components["mse_loss"] + components["r_loss"]
    if GRAD_LOSS_WEIGHT > 0:
        components["gradient_loss"] = GRAD_LOSS_WEIGHT * masked_gradient_l1(
            prediction,
            target,
            mask
        )
        loss = loss + components["gradient_loss"]
    if return_components:
        return loss, components
    return loss


def new_r_moments():
    return torch.zeros(6, device=device)


def add_r_moments(moments, prediction, target, mask):
    prediction = prediction.detach()[mask.bool()].float()
    target = target[mask.bool()].float()
    update = torch.stack(
        (
            prediction.new_full((), float(prediction.numel())),
            prediction.sum(),
            target.sum(),
            (prediction * prediction).sum(),
            (target * target).sum(),
            (prediction * target).sum()
        )
    )
    return moments + update


def r_from_moments(moments):
    moments = moments.reshape(6)
    n = moments[0]
    sum_x = moments[1]
    sum_y = moments[2]
    sum_xx = moments[3]
    sum_yy = moments[4]
    sum_xy = moments[5]
    cov = n * sum_xy - sum_x * sum_y
    var_x = (n * sum_xx - sum_x * sum_x).clamp_min(1e-8)
    var_y = (n * sum_yy - sum_y * sum_y).clamp_min(1e-8)
    return (cov / (var_x.sqrt() * var_y.sqrt())).item()


In [ ]:
%load_ext tensorboard
from torch.utils.tensorboard import SummaryWriter

model = UNet25D().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=4,
    min_lr=1e-5
)
EPOCHS = 75
EARLY_STOPPING_PATIENCE = 12
VALIDATION_EVERY = 1
L1_WEIGHT = 1.0
R_LOSS_WEIGHT = 0.5
GRAD_LOSS_WEIGHT = 0.3
RUN_NAME = "gradient-l1"
RESUME = False
CHECKPOINT_PATH = "best_outpainting_unet.pt"
RUN_STAMP = time.strftime("%Y-%m-%d_%H-%M-%S")
LOG_DIR = f"runs/outpainting/{RUN_STAMP}"
writer = SummaryWriter(LOG_DIR)
best_val_r = -float("inf")
start_epoch = 1
epochs_without_improvement = 0
history = {
    "train_loss": [],
    "val_loss": [],
    "train_r": [],
    "val_r": []
}

if RESUME:
    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location=device,
        weights_only=False
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    start_epoch = checkpoint["epoch"] + 1
    best_val_r = checkpoint["best_val_r"]
    history = checkpoint["history"]
    epochs_without_improvement = checkpoint["epochs_without_improvement"]
    random.setstate(checkpoint["python_random_state"])
    np.random.set_state(checkpoint["numpy_random_state"])
    torch.set_rng_state(checkpoint["torch_random_state"])
    if torch.cuda.is_available() and checkpoint["cuda_random_state"] is not None:
        torch.cuda.set_rng_state_all(checkpoint["cuda_random_state"])

if MLFLOW_AVAILABLE:
    if mlflow.active_run() is not None:
        mlflow.end_run()
    mlflow.start_run(run_name=f"{RUN_NAME}-epoch-{start_epoch}")
    mlflow.log_params(
        {
            "seed": SEED,
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "validation_every": VALIDATION_EVERY,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "l1_weight": L1_WEIGHT,
            "r_loss_weight": R_LOSS_WEIGHT,
            "grad_loss_weight": GRAD_LOSS_WEIGHT,
            "base_channels": 24,
            "dynamic_train_sampling": train_dataset.dynamic,
            "train_samples_per_center": TRAIN_SAMPLES_PER_CENTER,
            "amp_scale_range": str(AMP_SCALE_RANGE),
            "noise_sigma": NOISE_SIGMA,
            "noise_bank_size": NOISE_BANK_SIZE,
            "patch": f"{PATCH_H}x{PATCH_W}",
            "mask": f"2x{MASK_W}",
            "param_count": sum(p.numel() for p in model.parameters()),
        }
    )
    mlflow.set_tag("model/architecture", str(model))
    mlflow.set_tag("log_dir", LOG_DIR)


In [ ]:
def train_epoch():
    model.train()
    total_loss = torch.zeros((), device=device)
    component_totals = torch.zeros(4, device=device)
    r_moments = new_r_moments()

    for inputs, target, mask in tqdm(train_loader, desc="train", leave=False):
        inputs = inputs.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        mask = mask.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=use_amp):
            prediction = model(inputs)
            loss, components = reconstruction_loss(
                prediction,
                target,
                mask,
                return_components=True
            )

        loss.backward()
        optimizer.step()

        batch_size = inputs.size(0)
        total_loss += loss.detach().float() * batch_size
        component_values = torch.stack([
            value.detach().float() for value in components.values()
        ])
        component_totals += component_values * batch_size
        r_moments = add_r_moments(r_moments, prediction, target, mask)

    train_r = r_from_moments(r_moments)
    component_totals = component_totals / len(train_loader.dataset)
    loss_components = dict(zip(components, component_totals.cpu().tolist()))

    return (total_loss / len(train_loader.dataset)).item(), train_r, loss_components


In [ ]:
def validation_epoch():
    model.eval()
    total_loss = torch.zeros((), device=device)
    component_totals = torch.zeros(4, device=device)
    r_moments = new_r_moments()

    with torch.inference_mode():
        for inputs, target, mask in tqdm(val_loader, desc="val", leave=False):
            inputs = inputs.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)
            mask = mask.to(device, non_blocking=True)

            with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=use_amp):
                prediction = model(inputs)
                loss, components = reconstruction_loss(
                    prediction,
                    target,
                    mask,
                    return_components=True
                )

            batch_size = inputs.size(0)
            total_loss += loss.float() * batch_size
            component_values = torch.stack([
                value.detach().float() for value in components.values()
            ])
            component_totals += component_values * batch_size
            r_moments = add_r_moments(r_moments, prediction, target, mask)

    val_r = r_from_moments(r_moments)
    component_totals = component_totals / len(val_loader.dataset)
    loss_components = dict(zip(components, component_totals.cpu().tolist()))

    return (total_loss / len(val_loader.dataset)).item(), val_r, loss_components


In [ ]:
pbar = tqdm(range(start_epoch, EPOCHS + 1))
for epoch in pbar:
    train_loss, train_r, train_loss_components = train_epoch()

    history["train_loss"].append(train_loss)
    history["train_r"].append(train_r)
    writer.add_scalar("Loss/train", train_loss, epoch)
    writer.add_scalar("R/train", train_r, epoch)
    writer.add_scalar("Learning rate", optimizer.param_groups[0]["lr"], epoch)
    writer.add_scalar("Epoch", epoch, epoch)

    if MLFLOW_AVAILABLE:
        mlflow.log_metric("lr", optimizer.param_groups[0]["lr"], step=epoch)
        mlflow.log_metrics(
            {f"train_{name}": value for name, value in train_loss_components.items()},
            step=epoch
        )

    postfix = {"train_loss": f"{train_loss:.4f}", "train_R": f"{train_r:.4f}"}

    if epoch % VALIDATION_EVERY == 0:
        val_loss, val_r, val_loss_components = validation_epoch()
        scheduler.step(val_r)

        history["val_loss"].append(val_loss)
        history["val_r"].append(val_r)
        writer.add_scalar("Loss/validation", val_loss, epoch)
        writer.add_scalar("R/validation", val_r, epoch)

        if val_r > best_val_r:
            best_val_r = val_r
            epochs_without_improvement = 0
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "mean": mean,
                    "std": std,
                    "epoch": epoch,
                    "best_val_r": best_val_r,
                    "history": history,
                    "epochs_without_improvement": epochs_without_improvement,
                    "python_random_state": random.getstate(),
                    "numpy_random_state": np.random.get_state(),
                    "torch_random_state": torch.get_rng_state(),
                    "cuda_random_state": (
                        torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
                    ),
                    "config": {
                        "architecture": "UNet25D",
                        "base_channels": 24,
                                                "epochs": EPOCHS,
                        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
                        "l1_weight": L1_WEIGHT,
                        "r_loss_weight": R_LOSS_WEIGHT,
                        "grad_loss_weight": GRAD_LOSS_WEIGHT,
                        "learning_rate": 2e-3
                    }
                },
                CHECKPOINT_PATH
            )
        else:
            epochs_without_improvement += 1

        writer.add_scalar("R/best_validation", best_val_r, epoch)
        postfix["val_loss"] = f"{val_loss:.4f}"
        postfix["val_R"] = f"{val_r:.4f}"

        if MLFLOW_AVAILABLE:
            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("train_r", train_r, step=epoch)
            mlflow.log_metric("val_loss", val_loss, step=epoch)
            mlflow.log_metric("val_r", val_r, step=epoch)
            mlflow.log_metric("best_val_r", best_val_r, step=epoch)
            mlflow.log_metrics(
                {f"val_{name}": value for name, value in val_loss_components.items()},
                step=epoch
            )

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            pbar.set_postfix(postfix)
            break

    writer.flush()
    pbar.set_postfix(postfix)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)
model.load_state_dict(checkpoint["model_state_dict"])
best_epoch = checkpoint["epoch"]
writer.close()


In [ ]:
train_epochs = range(1, len(history["train_loss"]) + 1)
val_epochs = np.arange(
    VALIDATION_EVERY,
    len(history["val_loss"]) * VALIDATION_EVERY + 1,
    VALIDATION_EVERY
)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_epochs, history["train_loss"], label="Train")
axes[0].plot(val_epochs, history["val_loss"], label="Validation")
axes[0].set(title="Функция потерь", xlabel="Эпоха", ylabel="Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(train_epochs, history["train_r"], label="Train")
axes[1].plot(val_epochs, history["val_r"], label="Validation")
axes[1].set(title="Коэффициент корреляции", xlabel="Эпоха", ylabel="R")
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.tight_layout()
fig.savefig("outpainting_training_history.png", dpi=150, bbox_inches="tight")
display(Image("outpainting_training_history.png"))



## Блок "Тестирование модели"

Модель загружается из чекпоинта `best_outpainting_unet.pt` (лучшая по валидационному R) и оценивается на тестовой выборке: средняя ошибка, коэффициент R и примеры предсказаний. При активном MLflow-запуске тестовые метрики и модель записываются на трекер, модель регистрируется в Model Registry, после чего запуск закрывается.


In [ ]:
model = UNet25D().to(device)
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()


In [ ]:
model.eval()
test_loss = torch.zeros((), device=device)
prediction_values = []
target_values = []

with torch.inference_mode():
    for inputs, target, mask in test_loader:
        inputs = inputs.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        mask = mask.to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=use_amp):
            prediction = model(inputs)
            loss = reconstruction_loss(prediction, target, mask)

        test_loss += loss.float() * inputs.size(0)
        prediction_values.append(prediction[mask.bool()].float().cpu())
        target_values.append(target[mask.bool()].float().cpu())

test_loss = (test_loss / len(test_loader.dataset)).item()
test_predictions = torch.cat(prediction_values)
test_targets = torch.cat(target_values)
test_r = correlation_r(test_predictions, test_targets).item()

best_train_r = history["train_r"][best_epoch - 1] if history["train_r"] else 0
best_val_r = history["val_r"][best_epoch // VALIDATION_EVERY - 1] if history["val_r"] else 0
print(f"Test loss: {test_loss:.6f} | Test R: {test_r:.4f}")
print(
    f"Итоговые R | train: {best_train_r:.4f} | "
    f"val: {best_val_r:.4f} | test: {test_r:.4f}"
)

test_writer = SummaryWriter(LOG_DIR)
test_writer.add_scalar("Loss/test", test_loss, best_epoch)
test_writer.add_scalar("R/test", test_r, best_epoch)
test_writer.close()

if MLFLOW_AVAILABLE and mlflow.active_run() is not None:
    mlflow.log_metric("test_loss", test_loss)
    mlflow.log_metric("test_r", test_r)
    model_info = mlflow.pytorch.log_model(
        model,
        artifact_path=MLFLOW_MODEL_NAME,
        serialization_format="pickle",
    )
    try:
        MlflowClient().create_registered_model(MLFLOW_MODEL_NAME)
    except MlflowException:
        pass
    MlflowClient().create_model_version(
        name=MLFLOW_MODEL_NAME,
        source=model_info.model_uri,
        run_id=mlflow.active_run().info.run_id,
        tags={"best-val-r": str(best_val_r)},
    )
    mlflow.end_run()


In [ ]:
inputs, target, mask = next(iter(test_loader))
inputs = inputs.to(device)
target = target.to(device)
mask = mask.to(device)

with torch.inference_mode():
    with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=use_amp):
        prediction = model(inputs)

examples = min(3, inputs.size(0))
fig, axes = plt.subplots(examples, 3, figsize=(15, 5 * examples))
titles = ("Оригинал", "Восстановленный срез", "Абсолютная ошибка")

for index in range(examples):
    original = target[index, 0].cpu().numpy() * std + mean
    predicted = prediction[index, 0].float().cpu().numpy() * std + mean
    sample_mask = mask[index, 0].cpu().numpy().astype(bool)

    restored = original.copy()
    restored[sample_mask] = predicted[sample_mask]

    error = np.full_like(original, np.nan)
    error[sample_mask] = np.abs(predicted[sample_mask] - original[sample_mask])

    a_min, a_max = np.percentile(original, [1, 99])
    panels = (
        (original, "gray", a_min, a_max),
        (restored, "gray", a_min, a_max),
        (error, "hot", None, None),
    )
    for column, (data, colormap, vmin, vmax) in enumerate(panels):
        axis = axes[index, column]
        axis.imshow(data, cmap=colormap, vmin=vmin, vmax=vmax, aspect="auto")
        axis.set_xticks([])
        axis.set_yticks([])

    axes[index, 0].set_ylabel(f"Тестовый пример {index + 1}")
    if index == 0:
        for column, title in enumerate(titles):
            axes[0, column].set_title(title)

fig.tight_layout()
fig.savefig("outpainting_test_examples.png", dpi=150, bbox_inches="tight")
display(Image("outpainting_test_examples.png"))


## Блок "Инференса модели"

Сеть полносвёрточная и использует локальные признаки, поэтому после обучения на патчах 368×256 применяется ко всему срезу 481×800. Вход дополняется до размеров, кратных 16, а результат обрезается до исходной формы.

На срезах 0, 1 и двух последних индексах несуществующие контекстные кадры дублируют ближайший реальный кадр вместо нулей: срез 0 видит [f0, f0, f0, f1, f2], срез 459 — [f457, f458, f459, f459, f459].

Заполнение граничных полос выполняется за 1 проход: все пропущенные значения кодируются нулём и mask=1, как в обучении, а в куб записывается весь результат по маске пропусков. Соседние кадры каждого среза берутся из исходного куба, накопления между срезами нет. Визуализация отдельных срезов использует `restore_slice` — тот же проход.


In [ ]:
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

mask_0 = real_mask[0, :, :, 0] if real_mask.ndim == 4 else real_mask[0]
band_rect = np.zeros(cube.shape[1:], dtype=bool)
band_rect[0:24] = True
band_rect[457:481] = True
assert np.array_equal(band_rect, mask_0)


def predict_frames(frames, mask):
    frames = ((frames - mean) / std).astype(np.float32)
    mask = mask.astype(np.float32)
    frames *= (1 - mask)[None]

    inputs = np.concatenate((frames, mask[None]), axis=0)[None]
    inputs = torch.from_numpy(inputs).float().to(device)

    with torch.inference_mode():
        with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=use_amp):
            prediction = model(inputs)

    return prediction[0, 0].float().cpu().numpy() * std + mean


def predict_region(index):
    indices = np.clip(np.arange(index - 2, index + 3), 0, cube.shape[0] - 1)
    frames = np.stack([cube[i] for i in indices])
    return predict_frames(frames, real_mask[index])


def restore_slice(index):
    restored = cube[index].copy()
    prediction = predict_region(index)
    restored[real_mask[index]] = prediction[real_mask[index]]
    return restored


In [ ]:
def slice_panels(index):
    restored = restore_slice(index)
    original = cube[index].copy()
    missing = real_mask[index]

    original_display = original.copy()
    original_display[missing] = np.nan

    restored_region = np.full_like(restored, np.nan)
    restored_region[missing] = restored[missing]

    known = original[~missing]
    a_min, a_max = np.percentile(known, [1, 99])
    return original_display, restored_region, restored, a_min, a_max


def show_restored_slice(index):
    original_display, restored_region, restored, a_min, a_max = slice_panels(index)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    panels = (
        (original_display, "Исходный срез"),
        (restored_region, "Восстановленная область"),
        (restored, "Результат"),
    )
    for column, (data, title) in enumerate(panels):
        axes[column].imshow(data, cmap="gray", vmin=a_min, vmax=a_max, aspect="auto")
        axes[column].set_title(title)
        axes[column].set_xticks([])
        axes[column].set_yticks([])

    fig.suptitle(f"Срез {index}")
    fig.tight_layout()
    fig.savefig("outpainting_restored_slice.png", dpi=150, bbox_inches="tight")
    display(Image("outpainting_restored_slice.png"))


In [ ]:
indices = np.linspace(0, cube.shape[0] - 1, 4).astype(int)
panels = [slice_panels(index) for index in indices]

titles = ("Исходный срез", "Восстановленная область", "Результат")
fig, axes = plt.subplots(len(indices), 3, figsize=(15, 5 * len(indices)))

for row, index in enumerate(indices):
    original_display, restored_region, restored, a_min, a_max = panels[row]
    for column, (data, title) in enumerate(zip((original_display, restored_region, restored), titles)):
        axis = axes[row, column]
        axis.imshow(data, cmap="gray", vmin=a_min, vmax=a_max, aspect="auto")
        axis.set_xticks([])
        axis.set_yticks([])
    axes[row, 0].set_ylabel(f"Срез {index}")
    if row == 0:
        for column, title in enumerate(titles):
            axes[0, column].set_title(title)

fig.tight_layout()
fig.savefig("outpainting_restored_slices.png", dpi=150, bbox_inches="tight")
display(Image("outpainting_restored_slices.png"))


## Блок сохранения артефактов

Код и информация о том, как сохранялась модель и восстановленный куб (shape финального куба должен совпадать с shape'ом входного куба, который был скачан по ссылке)

In [ ]:
restored_cube = cube.copy()

for index in tqdm(range(cube.shape[0]), desc="inference"):
    prediction = predict_region(index)
    restored_cube[index, real_mask[index]] = prediction[real_mask[index]]

assert restored_cube.shape == cube.shape

output_path = "outpainting_cube_restored.npy"
model_output_path = "outpainting_unet_inference.pt"
np.save(output_path, restored_cube)
torch.save(model.state_dict(), model_output_path)
print(f"Восстановленный куб сохранен: {output_path}")
print(f"Модель для инференса сохранена: {model_output_path}")
